# Predicting the Cytotoxicity of Metal-Oxide Nanoparticles with Machine Learning

**Portfolio edition of the final computational notebook**

This notebook demonstrates the complete analytical workflow developed by Hassan Tariq: EDA, missing-data analysis, leakage-safe target definition, preprocessing, classification, viability regression, study-grouped validation, threshold analysis and comparative scientific interpretation.

The original embedded outputs have been removed to keep the repository compact and reproducible. Reported results are documented in the repository README and `docs/results-and-interpretation.md`.

**Research boundary:** this workflow supports screening and hypothesis generation. It is not a clinical or laboratory decision system.


## Reproducibility setup

A fixed random seed is defined and reused in model development and validation. Library versions are recorded so that the analysis can be reproduced later.


In [ ]:
from pathlib import Path
import sys
import platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("Random seed:", RANDOM_SEED)


In [ ]:
# Presentation theme: original results, one copy each, with a consistent dark style.
from cycler import cycler
from IPython.display import HTML

NOTEBOOK_BG = "#071015"
PANEL_BG = "#10191f"
GRID_COLOR = "#2a3c44"
TEXT_COLOR = "#eef6f6"
MUTED_COLOR = "#9bb0b5"
CHART_COLORS = ["#43d6bd", "#6f8cff", "#ffb45c", "#ff706e", "#c98cff"]

plt.style.use("dark_background")
matplotlib.rcParams.update({
    "figure.facecolor": NOTEBOOK_BG,
    "savefig.facecolor": NOTEBOOK_BG,
    "axes.facecolor": PANEL_BG,
    "axes.edgecolor": GRID_COLOR,
    "axes.labelcolor": TEXT_COLOR,
    "axes.titlecolor": TEXT_COLOR,
    "axes.grid": True,
    "grid.color": GRID_COLOR,
    "grid.alpha": 0.45,
    "text.color": TEXT_COLOR,
    "xtick.color": MUTED_COLOR,
    "ytick.color": MUTED_COLOR,
    "legend.facecolor": PANEL_BG,
    "legend.edgecolor": GRID_COLOR,
    "axes.prop_cycle": cycler(color=CHART_COLORS),
})

display(HTML(f"""
<style>
  .output_html, .jp-RenderedHTMLCommon {{ color: {TEXT_COLOR}; }}
  table.dataframe {{
    background: {PANEL_BG} !important;
    color: {TEXT_COLOR} !important;
    border-collapse: collapse !important;
    width: 100%;
  }}
  table.dataframe thead th {{
    background: #13242b !important;
    color: #62dfc8 !important;
    border: 1px solid #36515d !important;
    padding: 7px 9px !important;
  }}
  table.dataframe tbody th {{
    background: #0c171c !important;
    color: {MUTED_COLOR} !important;
    border: 1px solid #263942 !important;
  }}
  table.dataframe tbody td {{
    background: {PANEL_BG} !important;
    color: {TEXT_COLOR} !important;
    border: 1px solid #263942 !important;
    padding: 7px 9px !important;
  }}
  table.dataframe tbody tr:hover td {{ background: #183039 !important; }}
</style>
"""))

def add_bar_value_labels(ax, fmt="%.0f", padding=3):
    """Place exact values on every bar container without creating another chart."""
    for container in ax.containers:
        if hasattr(container, "datavalues"):
            ax.bar_label(container, fmt=fmt, padding=padding, fontsize=8, color=TEXT_COLOR)

def label_line_ends(ax, digits=3):
    """Direct-label the final point of each named line."""
    for line in ax.lines:
        label = line.get_label()
        if label.startswith("_") or len(line.get_xdata()) == 0:
            continue
        x_value = line.get_xdata()[-1]
        y_value = line.get_ydata()[-1]
        ax.annotate(
            f"{label} {y_value:.{digits}f}",
            xy=(x_value, y_value), xytext=(6, 0), textcoords="offset points",
            color=line.get_color(), fontsize=8, va="center"
        )
    ax.margins(x=0.16)

def annotate_sample_count(ax, count):
    ax.text(
        0.98, 0.03, f"n = {count:,}", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=8, color=MUTED_COLOR,
        bbox=dict(boxstyle="round,pad=0.3", facecolor=NOTEBOOK_BG, edgecolor=GRID_COLOR)
    )


## Load the two dataset versions

Version I and Version II are two processing stages of the same underlying literature-derived nanotoxicology resource, not two independent datasets. They are loaded separately and analysed with a common structure so that differences can be interpreted as effects of curation/processing rather than external validation.


In [ ]:
DATA_DIR = Path("../data")

FILE_V1 = DATA_DIR / "version_i.xlsx"
FILE_V2 = DATA_DIR / "version_ii.xlsx"

if not FILE_V1.exists() or not FILE_V2.exists():
    raise FileNotFoundError(
        "Place authorized dataset copies at data/version_i.xlsx and "
        "data/version_ii.xlsx. See data/README.md. "
        "For a self-contained example, run synthetic_demo_pipeline.ipynb."
    )

df_v1 = pd.read_excel(FILE_V1)
df_v2 = pd.read_excel(FILE_V2)

print("Version I shape:", df_v1.shape)
print("Version II shape:", df_v2.shape)

display(df_v1.head())


# Task A — Exploratory Data Analysis

The purpose of EDA is not only to describe the data. It is used to decide how the data should be cleaned and preprocessed before modelling.

The analysis therefore examines:

- dataset size and source-study structure;
- toxic/nontoxic class balance;
- material representation;
- all candidate numerical predictors, including physicochemical, electronic/electrical and exposure-related features;
- all candidate categorical predictors and their cardinality/frequency;
- skewness, ranges and possible extreme values;
- missingness and category consistency.

At the end of Task A, the observations are converted into explicit preprocessing decisions.


In [ ]:
def dataset_summary(df, name):
    toxicity_counts = df["Toxicity"].value_counts(dropna=False)
    toxic_n = toxicity_counts.get("Toxic", 0)
    return {
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Source studies": df["Pubmed ID"].nunique(),
        "Material types": df["Material type"].nunique(),
        "Toxic rows": int(toxic_n),
        "Toxic %": round(100 * toxic_n / len(df), 2),
        "Mean quality score": round(df["Average score"].mean(), 3),
        "Median quality score": round(df["Average score"].median(), 3),
    }

summary_table = pd.DataFrame([
    dataset_summary(df_v1, "Version I"),
    dataset_summary(df_v2, "Version II")
])

display(summary_table)


### A1. Class balance

The toxic class is the minority class, so raw accuracy could be misleading. This will influence metric selection in Task F; recall, F1 and threshold-independent/ranking metrics should be considered alongside accuracy.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
class_counts = pd.DataFrame({
    "Version I": df_v1["Toxicity"].value_counts(),
    "Version II": df_v2["Toxicity"].value_counts()
}).fillna(0)
class_counts.T.plot(kind="bar", ax=ax)
add_bar_value_labels(ax, fmt="%.0f")
ax.set_title("Toxicity Class Distribution")
ax.set_xlabel("Dataset")
ax.set_ylabel("Number of observations")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

display(class_counts)


### A2. Material representation

The literature dataset is heterogeneous, but some metal-oxide materials are represented much more frequently than others. This means model performance should not be interpreted as equally strong evidence for every material.


In [ ]:
material_counts_v1 = df_v1["Material type"].value_counts()
material_counts_v2 = df_v2["Material type"].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
material_counts_v1.head(12).sort_values().plot(kind="barh", ax=ax, color=CHART_COLORS[0])
add_bar_value_labels(ax, fmt="%.0f")
ax.set_title("Most Frequent Material Types — Version I")
ax.set_xlabel("Number of observations")
ax.set_ylabel("Material type")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "Version I count": material_counts_v1,
    "Version II count": material_counts_v2
}).fillna(0).sort_values("Version I count", ascending=False).head(15))


### A3. Observed toxicity rate by material

This is descriptive, not causal. A material's observed toxicity rate is confounded by dose, exposure duration, cell line, assay and other conditions. The plot is therefore used to identify patterns worth modelling, not to label a material as inherently safe or toxic.


In [ ]:
def toxicity_rate_by_material(df, min_n=30):
    temp = df.assign(is_toxic=(df["Toxicity"] == "Toxic").astype(int))
    grouped = temp.groupby("Material type").agg(
        n=("is_toxic", "size"),
        toxic_rate=("is_toxic", "mean")
    )
    return grouped[grouped["n"] >= min_n].sort_values("toxic_rate", ascending=False)

tox_by_material_v1 = toxicity_rate_by_material(df_v1, min_n=30)

fig, ax = plt.subplots(figsize=(10, 5))
(tox_by_material_v1["toxic_rate"] * 100).sort_values().plot(kind="barh", ax=ax, color=CHART_COLORS[3])
add_bar_value_labels(ax, fmt="%.1f")
ax.set_title("Observed Toxicity Rate by Material — Version I (n ≥ 30)")
ax.set_xlabel("Toxic observations (%)")
ax.set_ylabel("Material type")
plt.tight_layout()
plt.show()

display((tox_by_material_v1.assign(toxic_percent=tox_by_material_v1["toxic_rate"]*100)
         [["n", "toxic_percent"]].round(2)))


### A4. Full numerical feature-space analysis

The EDA examines the **full numerical feature space**, not only selected examples. The analysis below therefore evaluates every numerical predictor used for modelling across both dataset versions.

For each variable it reports:
- feature group (physicochemical, electronic/electrical, or exposure-related),
- missingness,
- minimum, quartiles, median, mean and maximum,
- skewness,
- whether the observed values are non-negative,
- and the scale/range of the feature.

The notebook also visualizes the distribution of **every numerical predictor**. This is important because preprocessing is not decided from a generic rule. A highly right-skewed non-negative exposure variable may justify a log transformation, whereas a signed electrical variable such as surface charge should not automatically be log-transformed.

Finally, pairwise numerical correlations are inspected because strong correlation among descriptors is relevant to the later choice of a regularized linear model (Ridge) for cell-viability regression.


In [ ]:
# Candidate numerical predictors available before exposure-time conversion.
raw_numeric_predictors = [
    "Core size (nm)", "Hydro size (nm)", "Surface charge (mV)",
    "Surface area (m2/g)", "ΔHsf (eV)", "Ec (eV)", "Ev (eV)",
    "χMeO (eV)", "Mass dose (ug/mL)"
]

feature_group_map = {
    "Core size (nm)": "Physicochemical",
    "Hydro size (nm)": "Physicochemical",
    "Surface charge (mV)": "Physicochemical",
    "Surface area (m2/g)": "Physicochemical",
    "ΔHsf (eV)": "Electronic/electrical",
    "Ec (eV)": "Electronic/electrical",
    "Ev (eV)": "Electronic/electrical",
    "χMeO (eV)": "Electronic/electrical",
    "Mass dose (ug/mL)": "Exposure",
}

def numeric_eda_table(df, version):
    rows = []
    for col in raw_numeric_predictors:
        s = pd.to_numeric(df[col], errors="coerce")
        q1, q3 = s.quantile([0.25, 0.75])
        rows.append({
            "Dataset": version,
            "Feature": col,
            "Group": feature_group_map[col],
            "Missing %": 100*s.isna().mean(),
            "Min": s.min(),
            "Q1": q1,
            "Median": s.median(),
            "Mean": s.mean(),
            "Q3": q3,
            "Max": s.max(),
            "Range": s.max() - s.min(),
            "Skewness": s.skew(),
            "Non-negative": bool((s.dropna() >= 0).all()),
        })
    return pd.DataFrame(rows)

numeric_eda = pd.concat([
    numeric_eda_table(df_v1, "Version I"),
    numeric_eda_table(df_v2, "Version II")
], ignore_index=True)
display(numeric_eda.round(3))

# Visualize every numerical predictor in both versions.
for col in raw_numeric_predictors:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=False)
    for ax, (df, version) in zip(
        axes, [(df_v1, "Version I"), (df_v2, "Version II")]
    ):
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        ax.hist(s, bins=30, color=CHART_COLORS[0], edgecolor=NOTEBOOK_BG, alpha=0.88)
        median_value = s.median()
        ax.axvline(median_value, color=CHART_COLORS[2], linestyle="--", linewidth=1.5, label=f"Median {median_value:.2f}")
        annotate_sample_count(ax, len(s))
        ax.legend(fontsize=7)
        ax.set_title(f"{version}: {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

# Correlation diagnostics among numerical predictors.
# These are target-independent EDA diagnostics, not model fitting.
for df, version in [(df_v1, "Version I"), (df_v2, "Version II")]:
    corr = (
        df[raw_numeric_predictors]
        .apply(pd.to_numeric, errors="coerce")
        .corr()
    )
    print(f"\n{version} numerical correlation matrix")
    display(corr.round(2))

    # Show the strongest absolute off-diagonal correlations.
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .rename("correlation")
        .reset_index()
        .rename(columns={"level_0": "Feature 1", "level_1": "Feature 2"})
    )
    pairs["abs_correlation"] = pairs["correlation"].abs()
    print(f"{version} strongest numerical correlations")
    display(
        pairs.sort_values("abs_correlation", ascending=False)
             .head(10)
             .round(3)
    )


### A5. Full categorical feature-space analysis

Categorical predictors can have very different cardinalities. For example, material type and cell type contain relatively few categories, while cell name and assay can contain many. This affects encoding dimensionality and the treatment of rare or unseen categories.


In [ ]:
categorical_predictors_raw = [
    "Material type", "Assay", "Cell name", "Cell species",
    "Cell origin", "Cell type", "Exposure time"
]

def categorical_eda_table(df, version):
    rows = []
    for col in categorical_predictors_raw:
        vc = df[col].astype("string").str.strip().value_counts(dropna=False)
        rows.append({
            "Dataset": version,
            "Feature": col,
            "Unique categories": df[col].nunique(dropna=True),
            "Missing %": 100*df[col].isna().mean(),
            "Most common": str(vc.index[0]) if len(vc) else "",
            "Most common %": 100*vc.iloc[0]/len(df) if len(vc) else np.nan,
            "Categories with <10 rows": int((vc < 10).sum()),
        })
    return pd.DataFrame(rows)

categorical_eda = pd.concat([
    categorical_eda_table(df_v1, "Version I"),
    categorical_eda_table(df_v2, "Version II")
], ignore_index=True)
display(categorical_eda.round(2))

# Show the most frequent levels for every categorical predictor in each version.
for col in categorical_predictors_raw:
    top = pd.concat([
        df_v1[col].astype("string").str.strip().value_counts().head(10).rename("Version I"),
        df_v2[col].astype("string").str.strip().value_counts().head(10).rename("Version II")
    ], axis=1).fillna(0)
    print(f"\nTop categories — {col}")
    display(top)


### A6. EDA → preprocessing decisions

The EDA is used to make the following modelling decisions rather than ending as a descriptive exercise.

| EDA observation | Why it matters | Decision used later |
|---|---|---|
| Numerical predictors are measured on very different physical scales | Scale-sensitive models can give disproportionate influence to large-magnitude variables | Standardize numerical inputs for Logistic Regression, KNN and Ridge after any justified transformation |
| Some non-negative numerical variables are strongly right-skewed | Extreme right tails can dominate linear/distance-based models | Apply `log1p` **only** when the observed feature is non-negative and right-skewed according to the explicit Task D rule |
| Signed variables such as surface charge may contain negative values | `log1p` is not a generally valid transformation for signed scientific variables | Do not automatically log-transform signed variables |
| Categorical predictors contain different numbers of levels, including rare levels | Direct numeric coding would impose a false order and very rare levels can create sparse unstable columns | Impute missing categories explicitly, use one-hot encoding, and group infrequent categories |
| Predictor groups have different meanings/distributions | A single preprocessing operation is not scientifically appropriate for every feature | Use separate numerical, categorical, and model-specific preprocessing branches |
| Some numerical descriptors are correlated | Correlated predictors can destabilize ordinary least-squares coefficients | Use Ridge as the regularized linear regression baseline and compare it with nonlinear tree models |
| Toxicity is imbalanced | Accuracy can hide poor minority-class detection | Emphasize precision, recall, F1, ROC-AUC and PR-AUC in addition to accuracy |
| Multiple observations come from the same PubMed study | Random row-wise splitting can leak study-specific patterns | Keep each PubMed study in only one fold using group-aware cross-validation |

**Decision carried forward:** Task B now examines whether the observed missingness supports retaining, imputing or excluding each type of variable; Task D then implements the preprocessing choices inside leakage-safe pipelines.


# Task B — Missing-Value and Data-Quality Analysis

Missingness is examined before modelling so that imputation is a reasoned response to the data rather than a generic default.

Two kinds of variables are distinguished:

1. **Predictive scientific variables** — physicochemical, electronic/electrical, biological and exposure descriptors used by the models.
2. **Provenance/quality metadata** — measurement-method text fields and reliability/quality scores that describe how observations were generated rather than the biological exposure itself.

The decision for each variable is therefore to **retain, transform, impute, or exclude** based on its scientific role and observed missingness.


In [ ]:
def missingness_table(df):
    out = pd.DataFrame({
        "missing_n": df.isna().sum(),
        "missing_%": 100 * df.isna().mean()
    })
    return out.sort_values("missing_%", ascending=False)

missing_v1 = missingness_table(df_v1)
missing_v2 = missingness_table(df_v2)

missing_compare = missing_v1.join(
    missing_v2,
    lsuffix="_v1",
    rsuffix="_v2"
)

display(missing_compare.head(15))


In [ ]:
method_cols = [
    "Method core size",
    "Method hydro size",
    "Method surface charge",
    "Method surface area"
]

method_missing = pd.DataFrame({
    "Version I missing": df_v1[method_cols].isna().sum(),
    "Version II missing": df_v2[method_cols].isna().sum()
})

fig, ax = plt.subplots(figsize=(9, 4))
method_missing.plot(kind="bar", ax=ax)
ax.set_title("Missing Measurement-Method Metadata")
ax.set_xlabel("Metadata field")
ax.set_ylabel("Missing observations")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

display(method_missing)


### B1. Category consistency checks

Categorical labels are checked for capitalization and whitespace inconsistencies before encoding. Equivalent scientific categories should map to a single label.


In [ ]:
categorical_cols = [
    "Material type", "Assay", "Cell name", "Cell species",
    "Cell origin", "Cell type", "Exposure time", "Toxicity"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Version I unique:", df_v1[col].nunique(dropna=False))
    print(df_v1[col].value_counts(dropna=False).head(12))


### B2. Missingness → feature-handling decisions

Missing-value handling is linked directly to what is observed in Tasks A and B.

For each modelling predictor, the notebook considers:
1. **how much is missing,**
2. **whether the variable is numerical or categorical,**
3. **the observed numerical distribution/skewness or categorical structure,**
4. **whether the feature has a clear scientific role**, and
5. **whether removing it would discard useful information.**

The strategy is therefore not “fill every blank with one generic value.”

- **Numerical scientific predictors:** retained when scientifically meaningful. Median imputation is used inside each training fold because several numerical predictors are skewed and the median is less influenced by extreme values than the mean.
- **Categorical scientific predictors:** retained and missing values are represented as an explicit `Missing` category. This avoids pretending that an unrecorded category is necessarily the most common biological category. One-hot encoding is then used, with infrequent levels grouped.
- **Measurement-method metadata:** kept for data-quality interpretation but excluded from ordinary predictive features because it is provenance information and is inconsistently populated.
- **Reliability/quality scores:** used to describe curation quality, not as exposure predictors, because using a curation score as if it were a nanoparticle property would change the scientific meaning of the prediction problem.
- **PubMed ID:** never imputed or used as a predictor; it is retained only as the study grouping variable for validation.

All imputation is fitted **inside the training fold** of the model pipeline, so validation-fold information cannot influence the replacement values.


In [ ]:
# Predictor-level missingness and distribution context.
model_candidate_cols = raw_numeric_predictors + [
    "Material type", "Assay", "Cell name", "Cell species", "Cell origin", "Cell type"
]

predictor_missingness = pd.DataFrame({
    "Version I missing %": 100*df_v1[model_candidate_cols].isna().mean(),
    "Version II missing %": 100*df_v2[model_candidate_cols].isna().mean(),
})
display(predictor_missingness.sort_values("Version I missing %", ascending=False).round(2))

# Link numerical missingness to the observed distributions.
numeric_missing_context = []
for col in raw_numeric_predictors:
    for df, version in [(df_v1, "Version I"), (df_v2, "Version II")]:
        s = pd.to_numeric(df[col], errors="coerce")
        numeric_missing_context.append({
            "Dataset": version,
            "Feature": col,
            "Missing %": 100*s.isna().mean(),
            "Mean": s.mean(),
            "Median": s.median(),
            "Skewness": s.skew(),
            "Decision": "Retain; median-impute inside training folds"
        })
display(pd.DataFrame(numeric_missing_context).round(3))

# Explicit feature-group handling plan.
feature_handling = pd.DataFrame([
    ["Physicochemical numerical",
     "Core size, hydro size, surface charge, surface area",
     "Retain; median-impute; log only if data-supported; standardize for scale-sensitive models"],
    ["Electronic/electrical numerical",
     "ΔHsf, Ec, Ev, χMeO",
     "Retain; median-impute; do not force log transformation; standardize for scale-sensitive models"],
    ["Exposure numerical",
     "Mass dose, exposure time (hours)",
     "Retain; convert duration to hours; median-impute; log only if non-negative and right-skewed; standardize for scale-sensitive models"],
    ["Biological categorical",
     "Assay, cell name/species/origin/type",
     "Retain; explicit Missing level; one-hot encode; group infrequent levels"],
    ["Material categorical",
     "Material type",
     "Retain; explicit Missing level; one-hot encode; group infrequent levels"],
    ["PubMed study identifier",
     "Pubmed ID",
     "Use only for grouped validation; never as a predictor"],
    ["Measurement-method metadata",
     "Method core/hydro/surface charge/surface area",
     "Exclude from ordinary predictors; use for data-quality/provenance analysis"],
    ["Reliability/quality metadata",
     "Reliability and average-score fields",
     "Exclude from ordinary predictors; use for curation/data-quality interpretation"],
], columns=["Feature group", "Examples", "Handling decision"])

display(feature_handling)


### Task B takeaway

The missing-data analysis shows why the pipeline does **not** use one generic rule for every column. Scientific predictors are retained and handled according to type; provenance fields with heavy/inconsistent missingness are excluded; and leakage-prone endpoints are removed. These decisions feed directly into Task D.


# Task C — Define the Prediction Problems

Two separate supervised-learning problems are created.

## Classification

- **Target:** `Toxicity`
- **Forbidden predictor:** `Viability (%)`
- **Grouping variable for validation:** `Pubmed ID`

## Regression

- **Target:** `Viability (%)`
- **Forbidden predictor:** `Toxicity`
- **Grouping variable for validation:** `Pubmed ID`

The grouping variable is retained outside the model features so that source-study-aware validation can be implemented in Task F.


# Task D — Data Preparation

The preprocessing strategy now follows directly from Tasks A and B. Different feature groups receive different treatment because they have different distributions, meanings and modelling requirements.

### Predictive feature groups and preprocessing

| Feature group | Variables | Processing before modelling |
|---|---|---|
| **Physicochemical** | Core size, hydrodynamic size, surface charge, surface area | Median imputation if required; `log1p` only when non-negative and strongly right-skewed; standardization for linear/KNN models; no scaling for trees |
| **Electronic/electrical** | ΔHsf, Ec, Ev, χMeO | Median imputation; no blind log transform because signed values may occur; standardization for linear/KNN models; no scaling for trees |
| **Exposure** | Exposure time, mass dose | Convert exposure time to numeric hours; median imputation if required; log-transform only if non-negative and strongly skewed; standardize for linear/KNN models |
| **Biological categorical** | Assay, cell name, species, origin, type | Clean labels; explicit missing category; one-hot encoding; infrequent/unseen levels handled safely |
| **Material identity** | Material type | Clean labels; explicit missing category; one-hot encoding; infrequent/unseen levels handled safely |

### Excluded from ordinary predictors

- PubMed ID — validation grouping only
- measurement-method text fields — provenance metadata
- per-attribute reliability/data-quality scores and Average score — data-quality metadata
- alternate endpoint — excluded to prevent target leakage

This feature-group-specific design avoids the mistake of applying a single normalization strategy to every variable.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

NUMERIC_FEATURES = [
    "Core size (nm)",
    "Hydro size (nm)",
    "Surface charge (mV)",
    "Surface area (m2/g)",
    "ΔHsf (eV)",
    "Ec (eV)",
    "Ev (eV)",
    "χMeO (eV)",
    "Exposure time (hours)",
    "Mass dose (ug/mL)",
]

CATEGORICAL_FEATURES = [
    "Material type",
    "Assay",
    "Cell name",
    "Cell species",
    "Cell origin",
    "Cell type",
]

GROUP_COLUMN = "Pubmed ID"
CLASSIFICATION_TARGET = "Toxicity"
REGRESSION_TARGET = "Viability (%)"

print("Numeric predictors:", len(NUMERIC_FEATURES))
print("Categorical predictors:", len(CATEGORICAL_FEATURES))


### D1. Deterministic cleaning

Exposure duration is converted from strings such as `24h` into numeric hours. Text categories are stripped of excess whitespace, and known capitalization inconsistencies such as `human` versus `Human` are harmonized.

This function is applied identically to both versions.


In [ ]:
def clean_dataset(df):
    out = df.copy()

    # Convert exposure duration such as "24h" or "0.5h" to numeric hours.
    out["Exposure time (hours)"] = (
        out["Exposure time"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("hours", "", regex=False)
        .str.replace("hour", "", regex=False)
        .str.replace("hrs", "", regex=False)
        .str.replace("hr", "", regex=False)
        .str.replace("h", "", regex=False)
    )
    out["Exposure time (hours)"] = pd.to_numeric(
        out["Exposure time (hours)"], errors="coerce"
    )

    # Strip whitespace from relevant categorical variables.
    text_cols = [
        "Material type", "Assay", "Cell name", "Cell species",
        "Cell origin", "Cell type", "Toxicity"
    ]
    for col in text_cols:
        out[col] = out[col].where(out[col].isna(), out[col].astype(str).str.strip())

    # Harmonize known capitalization inconsistency.
    out["Cell species"] = out["Cell species"].replace({
        "human": "Human"
    })

    return out

clean_v1 = clean_dataset(df_v1)
clean_v2 = clean_dataset(df_v2)

print("Version I exposure-time missing after conversion:",
      clean_v1["Exposure time (hours)"].isna().sum())
print("Version II exposure-time missing after conversion:",
      clean_v2["Exposure time (hours)"].isna().sum())

display(clean_v1[["Exposure time", "Exposure time (hours)", "Cell species"]].head())


### D2. Evidence-based transformation, scaling and categorical encoding

The notebook distinguishes **transformation** from **scaling/normalization**.

- **Log transformation (`log1p`)** changes the *shape* of a distribution. It is considered only for predictors that are non-negative and show strong **right** skewness in the observed data.
- **Standardization (`StandardScaler`)** changes the *scale* of a numerical feature to zero mean and unit variance after imputation/transformation. This is important for Logistic Regression, KNN and Ridge because their coefficients/distances are sensitive to feature scale.
- **Random Forest and XGBoost** do not require numerical standardization because tree splits depend on ordering and thresholds rather than Euclidean distance or coefficient magnitude. They therefore receive imputed numerical values without `StandardScaler`.
- **Categorical variables** are not assigned arbitrary integer codes. Missing categories are made explicit, then one-hot encoding is used because the categories have no natural numeric ordering. `handle_unknown="infrequent_if_exist"` and `min_frequency=10` group rare levels and allow unseen validation-fold categories to be handled safely.

#### Rigorous log-transformation rule
For every numerical predictor the code reports skewness in Version I and Version II and whether all observed values are non-negative. A feature is log-transformed only when:

1. it is non-negative in both versions, **and**
2. skewness is greater than `+1` in at least one version.

The rule uses **positive** skewness, not absolute skewness: a log transformation is intended here to compress a long right tail, not to transform a left-skewed variable merely because its absolute skewness is large.

The code also reports before/after skewness for selected log features so that the transformation can be checked against the observed data rather than assumed to help.


In [ ]:
# Determine log-transform candidates from observed EDA, using a transparent rule.
# This decision is target-independent and documented here for reproducibility.
clean_numeric_for_eda = [
    "Core size (nm)", "Hydro size (nm)", "Surface charge (mV)",
    "Surface area (m2/g)", "ΔHsf (eV)", "Ec (eV)", "Ev (eV)",
    "χMeO (eV)", "Exposure time (hours)", "Mass dose (ug/mL)"
]

log_decision_rows = []
for col in clean_numeric_for_eda:
    s1 = pd.to_numeric(clean_v1[col], errors="coerce")
    s2 = pd.to_numeric(clean_v2[col], errors="coerce")
    nonnegative = bool((s1.dropna() >= 0).all() and (s2.dropna() >= 0).all())
    skew1, skew2 = s1.skew(), s2.skew()
    use_log = nonnegative and (skew1 > 1 or skew2 > 1)
    log_decision_rows.append({
        "Feature": col, "Version I skew": skew1, "Version II skew": skew2,
        "Non-negative in both": nonnegative, "Apply log1p": use_log
    })

log_decisions = pd.DataFrame(log_decision_rows)
display(log_decisions.round(3))

LOG_FEATURES = log_decisions.loc[log_decisions["Apply log1p"], "Feature"].tolist()
OTHER_NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c not in LOG_FEATURES]
print("Log-transformed numerical features:", LOG_FEATURES)
print("Other numerical features:", OTHER_NUMERIC_FEATURES)

log_numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler()),
])

other_numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=10
    )),
])

# Scale-sensitive preprocessor used by Logistic Regression, Ridge and KNN.
preprocessor = ColumnTransformer(
    transformers=[
        ("log_numeric", log_numeric_pipeline, LOG_FEATURES),
        ("numeric", other_numeric_pipeline, OTHER_NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop"
)

print(preprocessor)


# Verify what the selected transformations do to skewness.
before_after_skew = []
for col in LOG_FEATURES:
    for df, version in [(clean_v1, "Version I"), (clean_v2, "Version II")]:
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        transformed = np.log1p(s)
        before_after_skew.append({
            "Dataset": version,
            "Feature": col,
            "Skew before": s.skew(),
            "Skew after log1p": pd.Series(transformed).skew(),
        })
if before_after_skew:
    display(pd.DataFrame(before_after_skew).round(3))


### D3. Create leakage-safe X, y and group objects

The alternate endpoint is never included in `X`. PubMed ID is retained separately as `groups` for study-aware validation.


In [ ]:
def make_problem_data(df):
    predictor_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES

    X = df[predictor_cols].copy()
    groups = df[GROUP_COLUMN].copy()

    y_class = df[CLASSIFICATION_TARGET].map({
        "Nontoxic": 0,
        "Toxic": 1
    })

    y_reg = df[REGRESSION_TARGET].astype(float)

    return X, y_class, y_reg, groups

X_v1, yclass_v1, yreg_v1, groups_v1 = make_problem_data(clean_v1)
X_v2, yclass_v2, yreg_v2, groups_v2 = make_problem_data(clean_v2)

print("Version I X:", X_v1.shape)
print("Version II X:", X_v2.shape)
print("Classification target values V1:", yclass_v1.value_counts().to_dict())
print("Regression target range V1:", (yreg_v1.min(), yreg_v1.max()))

assert "Viability (%)" not in X_v1.columns
assert "Toxicity" not in X_v1.columns
assert "Pubmed ID" not in X_v1.columns

print("Leakage checks passed.")


## Tasks A–D checkpoint

At this stage the analysis has moved from description to decisions:

- **Task A:** the full numerical, categorical, electronic/electrical and exposure feature space has been inspected; class imbalance and feature distributions now explicitly inform preprocessing.
- **Task B:** missingness is linked to retain/impute/exclude decisions rather than handled generically.
- **Task C:** classification and regression targets are defined with leakage safeguards.
- **Task D:** preprocessing is feature-group specific, with evidence-based log transforms, scaling for scale-sensitive models, categorical encoding, rare/unseen-category handling and training-fold-only fitting.

### Next: Task E — Model Development

The modelling stage begins with interpretable baselines, then tests whether additional model flexibility genuinely improves grouped cross-validation performance.


# Task E — Model Development: Classification

Four deliberately different classifier families are compared under the same study-aware validation principle:

1. **Logistic Regression** — interpretable linear baseline; scaled predictors; balanced class weights.
2. **K-Nearest Neighbours (KNN)** — simple distance-based reference model; scaling is essential because distance depends on feature magnitude.
3. **Random Forest** — bagged non-linear trees that capture interactions and do not require numerical standardization.
4. **XGBoost** — boosted non-linear trees with class-imbalance weighting.

KNN is included as a simple baseline specifically to test whether a distance-based approach benefits from the standardized feature space.

All models use **5-fold Stratified Group Cross-Validation** with `Pubmed ID` as the grouping variable. Rows from the same source study never appear in both the training and validation portion of a fold.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score
)
from xgboost import XGBClassifier

def tree_preprocessor():
    # Tree models need missing-value handling and categorical encoding,
    # but not numerical standardization.
    return ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
            ("categorical", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                ("onehot", OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    min_frequency=10
                )),
            ]), CATEGORICAL_FEATURES),
        ],
        remainder="drop"
    )

def evaluate_classification_models(X, y, groups, dataset_name):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    splits = list(cv.split(X, y, groups))
    rows, oof = [], {}

    for model_name in ["Logistic Regression", "KNN", "Random Forest", "XGBoost"]:
        oof_prob = np.full(len(X), np.nan)
        oof_fold = np.full(len(X), -1)

        for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            if model_name == "Logistic Regression":
                estimator = Pipeline(steps=[
                    ("preprocessor", preprocessor),
                    ("model", LogisticRegression(
                        max_iter=1000, solver="liblinear",
                        class_weight="balanced", random_state=RANDOM_SEED
                    ))
                ])
            elif model_name == "KNN":
                estimator = Pipeline(steps=[
                    ("preprocessor", preprocessor),
                    ("model", KNeighborsClassifier(
                        n_neighbors=15, weights="distance", n_jobs=1
                    ))
                ])
            elif model_name == "Random Forest":
                estimator = Pipeline(steps=[
                    ("preprocessor", tree_preprocessor()),
                    ("model", RandomForestClassifier(
                        n_estimators=80, min_samples_leaf=2,
                        class_weight="balanced", random_state=RANDOM_SEED,
                        n_jobs=1
                    ))
                ])
            else:
                n_negative = int((y_train == 0).sum())
                n_positive = int((y_train == 1).sum())
                scale_pos_weight = n_negative / max(n_positive, 1)
                estimator = Pipeline(steps=[
                    ("preprocessor", tree_preprocessor()),
                    ("model", XGBClassifier(
                        n_estimators=100, max_depth=4, learning_rate=0.06,
                        subsample=0.85, colsample_bytree=0.85,
                        objective="binary:logistic", eval_metric="logloss",
                        scale_pos_weight=scale_pos_weight, reg_lambda=1.0,
                        tree_method="hist", random_state=RANDOM_SEED, n_jobs=1
                    ))
                ])

            estimator.fit(X_train, y_train)
            prob = estimator.predict_proba(X_valid)[:, 1]
            pred = (prob >= 0.50).astype(int)
            oof_prob[valid_idx] = prob
            oof_fold[valid_idx] = fold

            rows.append({
                "Dataset": dataset_name, "Model": model_name, "Fold": fold,
                "Accuracy": accuracy_score(y_valid, pred),
                "Balanced Accuracy": balanced_accuracy_score(y_valid, pred),
                "Precision": precision_score(y_valid, pred, zero_division=0),
                "Recall": recall_score(y_valid, pred, zero_division=0),
                "F1": f1_score(y_valid, pred, zero_division=0),
                "ROC-AUC": roc_auc_score(y_valid, prob),
                "PR-AUC": average_precision_score(y_valid, prob),
            })

        oof[model_name] = {"prob": oof_prob, "fold": oof_fold}

    return pd.DataFrame(rows), oof

class_results_v1, class_oof_v1 = evaluate_classification_models(X_v1, yclass_v1, groups_v1, "Version I")
class_results_v2, class_oof_v2 = evaluate_classification_models(X_v2, yclass_v2, groups_v2, "Version II")
classification_results = pd.concat([class_results_v1, class_results_v2], ignore_index=True)
classification_summary = (
    classification_results.groupby(["Dataset", "Model"])[
        ["Accuracy", "Balanced Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
    ].agg(["mean", "std"]).round(3)
)
display(classification_summary)


## E1. Classification comparison and metric interpretation

Because the toxic class is the minority class, **accuracy alone can be misleading**. A model could classify many majority-class rows correctly while still missing toxic cases. The following metrics therefore answer different questions:

- **Precision:** among conditions predicted as toxic, what fraction are actually toxic? Low precision means many false alarms.
- **Recall (sensitivity):** among truly toxic conditions, what fraction are detected? Low recall means toxic cases are being missed.
- **F1-score:** harmonic mean of precision and recall. It is useful when both missed toxic cases and false alarms matter and a single threshold-dependent summary is needed.
- **ROC-AUC:** evaluates how well the model ranks toxic above nontoxic observations over all thresholds. It is threshold-independent, but can look optimistic when the positive class is rare.
- **PR-AUC (average precision):** focuses directly on precision and recall for the positive/toxic class, so it is especially informative under class imbalance. Its no-skill reference is related to the toxic-class prevalence rather than 0.5.
- **Balanced accuracy:** gives equal weight to sensitivity for each class and is therefore more informative than ordinary accuracy when classes are uneven.
- **Accuracy:** retained only as a familiar supplementary metric.

For this screening problem, **recall has particular practical importance** if the cost of overlooking a potentially toxic condition is high. F1 provides a balanced summary, while PR-AUC and ROC-AUC show ranking quality independently of one fixed threshold.


In [ ]:
mean_results = (
    classification_results
    .groupby(["Dataset", "Model"])[
        ["Accuracy", "Balanced Accuracy", "Precision", "Recall",
         "F1", "ROC-AUC", "PR-AUC"]
    ]
    .mean()
    .round(3)
)

display(mean_results)

# Numerical labels are placed directly on the bars so exact values are visible.
for metric in ["ROC-AUC", "PR-AUC", "Recall", "F1"]:
    plot_data = mean_results[metric].unstack("Model")
    ax = plot_data.plot(kind="bar", figsize=(9, 4))
    ax.set_title(f"Classification Model Comparison — {metric}")
    ax.set_ylabel(metric)
    ax.set_xlabel("Dataset version")
    ax.tick_params(axis="x", rotation=0)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
    plt.tight_layout()
    plt.show()

# Explicitly identify best models under different objectives.
for dataset in ["Version I", "Version II"]:
    d = mean_results.loc[dataset]
    print(f"\n{dataset}")
    for metric in ["F1", "Recall", "ROC-AUC", "PR-AUC"]:
        best_model = d[metric].idxmax()
        best_value = d.loc[best_model, metric]
        print(f"Best {metric}: {best_model} ({best_value:.3f})")


### Classification takeaway

Observed F1 values around **0.45 for Version I and 0.50 for Version II** are **modest**, not strong, and should not be presented as evidence that cytotoxicity is solved.

Plausible contributors are investigated throughout the notebook:
- **class imbalance:** toxic observations are less common, making minority-class learning harder;
- **feature informativeness:** the available descriptors may not capture all mechanisms governing toxicity;
- **missingness:** some scientific variables are incompletely reported in literature-derived data;
- **noise and heterogeneity:** studies use different laboratories, assays, cell systems and protocols;
- **preprocessing:** log transformation and scaling can help appropriate model families but cannot recover information that is not recorded;
- **rare categories:** some materials/cell systems have limited support;
- **study shifts:** group-aware validation deliberately tests on source studies absent from each training fold, which is harder and more realistic than random row-wise splitting.

A modest F1 under leakage-resistant grouped validation is therefore interpreted as a limitation of the current predictive signal, not hidden by reporting accuracy alone.


## Task E classification checkpoint

The classification comparison now contains four complementary model families:

- **Logistic Regression:** linear/interpretable, scale-sensitive baseline
- **KNN:** distance-based, scale-sensitive baseline
- **Random Forest:** bagged non-linear tree model
- **XGBoost:** boosted non-linear tree model

The next stage addresses the continuous cell-viability response rather than adding models without a clear purpose.


# Task E — Model Development: Cell-Viability Regression

The second prediction problem models continuous `Viability (%)`.

Three focused regression families are compared under source-study-aware validation:

1. **Ridge Regression** — regularized linear baseline.
2. **Random Forest Regressor** — nonlinear bagged-tree model.
3. **XGBoost Regressor** — nonlinear boosted-tree model.

### Why Ridge rather than ordinary least-squares linear regression?

Ridge is used as the principal linear baseline because this design contains:
- multiple related physicochemical/electronic numerical descriptors,
- categorical variables that expand into many one-hot encoded columns,
- and therefore a realistic risk that ordinary least-squares coefficients would become unstable when predictors are correlated or the encoded design is high-dimensional.

Ridge minimizes the usual squared-error objective **plus an L2 penalty on coefficient magnitude**. It does not solve weak signal or missing information, but it can stabilize the linear solution when predictors overlap in information.

The diagnostic cell below reports numerical correlations and the number of columns created by preprocessing. This makes the regularization rationale observable rather than merely asserted. Ridge is still evaluated empirically: if its cross-validated R² is poor or negative, the notebook states that clearly rather than treating regularization as automatically beneficial.


In [ ]:
from sklearn.base import clone
# Ridge-design diagnostics (target-independent).
# 1) Re-report strongest numerical predictor correlations.
for df, version in [(clean_v1, "Version I"), (clean_v2, "Version II")]:
    corr = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").corr()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    strongest = upper.stack().abs().sort_values(ascending=False).head(10)
    print(f"\n{version} — strongest absolute numerical correlations")
    display(strongest.rename("|correlation|").to_frame().round(3))

# 2) Show how many columns the one-hot/scaled design creates.
# This is a target-independent dimensionality diagnostic only; model evaluation
# remains strictly fold-wise in the CV pipelines below.
for X, version in [(X_v1, "Version I"), (X_v2, "Version II")]:
    temp_preprocessor = clone(preprocessor)
    Xt = temp_preprocessor.fit_transform(X)
    print(
        f"{version} — Ridge-design diagnostics: "
        f"{X.shape[1]} raw predictors -> {Xt.shape[1]} preprocessed columns"
    )


## Regression metrics and what they mean biologically

Regression metrics are interpreted in **cell-viability percentage points**, not reported as abstract numbers.

- **MAE:** average absolute prediction error. An MAE of about 20 means that the predicted viability differs from the observed viability by roughly **20 percentage points on average**. On a biologically interpreted percentage scale, that is a substantial error and limits use for precise prediction.
- **RMSE:** also measures error in percentage points but penalizes large misses more strongly than MAE. RMSE much larger than MAE indicates that some observations have especially large prediction errors.
- **R²:** fraction of observed variability explained relative to a simple mean-prediction reference. For example, R² = 0.20 means only about 20% of the observed viability variation is explained. **Negative R² means the model performs worse than predicting the mean viability for those validation observations.**

The code below additionally reports MAE relative to the observed target range and standard deviation. This prevents a number such as MAE = 20 from being interpreted without reference to the biological scale.


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

def evaluate_regression_models(X, y, groups, dataset_name):
    cv = GroupKFold(n_splits=5)
    splits = list(cv.split(X, y, groups))
    rows, oof = [], {}

    for model_name in ["Ridge Regression", "Random Forest", "XGBoost"]:
        oof_pred = np.full(len(X), np.nan)
        oof_fold = np.full(len(X), -1)

        for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            if model_name == "Ridge Regression":
                estimator = Pipeline([
                    ("preprocessor", preprocessor),
                    ("model", Ridge(alpha=1.0))
                ])
            elif model_name == "Random Forest":
                estimator = Pipeline([
                    ("preprocessor", tree_preprocessor()),
                    ("model", RandomForestRegressor(
                        n_estimators=30, min_samples_leaf=2,
                        random_state=RANDOM_SEED, n_jobs=1
                    ))
                ])
            else:
                estimator = Pipeline([
                    ("preprocessor", tree_preprocessor()),
                    ("model", XGBRegressor(
                        n_estimators=80, max_depth=4, learning_rate=0.06,
                        subsample=0.85, colsample_bytree=0.85,
                        objective="reg:squarederror", tree_method="hist",
                        reg_lambda=1.0, random_state=RANDOM_SEED, n_jobs=1
                    ))
                ])

            estimator.fit(X_train, y_train)
            pred = estimator.predict(X_valid)
            oof_pred[valid_idx] = pred
            oof_fold[valid_idx] = fold
            rows.append({
                "Dataset": dataset_name, "Model": model_name, "Fold": fold,
                "MAE": mean_absolute_error(y_valid, pred),
                "RMSE": mean_squared_error(y_valid, pred) ** 0.5,
                "R2": r2_score(y_valid, pred)
            })

        oof[model_name] = {"pred": oof_pred, "fold": oof_fold}

    return pd.DataFrame(rows), oof

reg_results_v1, reg_oof_all_v1 = evaluate_regression_models(X_v1, yreg_v1, groups_v1, "Version I")
reg_results_v2, reg_oof_all_v2 = evaluate_regression_models(X_v2, yreg_v2, groups_v2, "Version II")
regression_results = pd.concat([reg_results_v1, reg_results_v2], ignore_index=True)
regression_summary = (
    regression_results.groupby(["Dataset", "Model"])[["MAE", "RMSE", "R2"]]
    .agg(["mean", "std"]).round(3)
)
display(regression_summary)

In [ ]:
regression_means = (
    regression_results.groupby(["Dataset", "Model"])[["MAE", "RMSE", "R2"]]
    .mean().round(3)
)
display(regression_means)

for metric in ["MAE", "RMSE", "R2"]:
    ax = regression_means[metric].unstack("Model").plot(kind="bar", figsize=(9, 4))
    ax.set_title(f"Regression Model Comparison — {metric}")
    ax.set_xlabel("Dataset version")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=0)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
    plt.tight_layout()
    plt.show()

# Explicit Version I vs Version II deltas for each model.
reg_delta_rows = []
for model in regression_means.index.get_level_values("Model").unique():
    v1 = regression_means.loc[("Version I", model)]
    v2 = regression_means.loc[("Version II", model)]
    reg_delta_rows.append({
        "Model": model,
        "MAE change V2-V1": v2["MAE"] - v1["MAE"],
        "RMSE change V2-V1": v2["RMSE"] - v1["RMSE"],
        "R2 change V2-V1": v2["R2"] - v1["R2"],
    })
regression_version_change = pd.DataFrame(reg_delta_rows).set_index("Model").round(3)
# This comparison table is displayed once, in Task G where Version I and Version II are compared.


# Interpret MAE in relation to the biological target scale.
target_context_rows = []
for dataset, y in [("Version I", yreg_v1), ("Version II", yreg_v2)]:
    target_range = y.max() - y.min()
    target_sd = y.std()
    for model in regression_means.loc[dataset].index:
        mae = regression_means.loc[(dataset, model), "MAE"]
        target_context_rows.append({
            "Dataset": dataset,
            "Model": model,
            "MAE (percentage points)": mae,
            "Target min": y.min(),
            "Target max": y.max(),
            "Target range": target_range,
            "Target SD": target_sd,
            "MAE as % of target range": 100*mae/target_range if target_range else np.nan,
            "MAE / target SD": mae/target_sd if target_sd else np.nan,
        })
print("\nRegression error in target context")
display(pd.DataFrame(target_context_rows).round(3))


## Regression interpretation

The regression results are judged for **scientific usefulness**, not only mathematical validity.

- If MAE is near 15–20 percentage points, the model's typical error is large enough to matter biologically on a viability-percentage scale.
- R² values around **0.15–0.25** indicate that only a limited share of viability variability is captured by the available nanoparticle, biological and exposure descriptors.
- A **negative Ridge R²**, if observed, means that the regularized linear model generalizes worse than a simple mean-prediction baseline on those validation folds. This is evidence against assuming that the relationship is adequately represented by a stable linear model.

Plausible reasons for limited regression performance include heterogeneous laboratories and assays, unmeasured experimental factors, noisy literature-derived measurements, sparse or missing descriptors, nonlinear interactions, uneven material/cell-system representation, and source-study shifts between folds.

The Version I–Version II delta table is therefore interpreted explicitly:
- lower MAE/RMSE in Version II = improvement in prediction error,
- higher R² in Version II = improvement in explained variability,
- but small numerical changes are not automatically considered practically important.

Because Version I and Version II are processing stages of the same underlying resource, any improvement is interpreted as an association with curation/preprocessing differences—not as independent external validation.


# Task F — Validation

The objective of validation is to estimate whether the models can generalize to **observations from source studies that were not used to fit the model**.

## Exact out-of-fold (OOF) procedure

For each dataset version:

1. `Pubmed ID` defines the grouping variable.
2. Classification uses `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` so class balance is considered while keeping each PubMed study in only one fold.
3. Regression uses `GroupKFold(n_splits=5)` so every PubMed study remains entirely within one fold.
4. At each iteration, four folds of studies are used to fit the complete preprocessing + model pipeline.
5. The remaining fold is **not used during fitting**; predictions are generated for that validation fold.
6. This repeats until every observation has exactly one prediction from a model that did not train on its source-study fold.
7. The combined predictions are called **out-of-fold (OOF) cross-validation predictions**.

### Important terminology

There is **no completely separate independent held-out test set** in this notebook. Therefore, OOF predictions are **not** described as “held-out test-set predictions.” They are cross-validation estimates. A future independent external dataset would be required for a true final external test.

Keeping preprocessing inside each fold is essential: imputation, scaling and encoding are fitted on the training folds only, preventing validation information from leaking into model fitting.


In [ ]:
from sklearn.metrics import confusion_matrix

# Select the classifier with the highest mean F1 in each version for detailed
# threshold analysis. This choice is transparent and derives from Task E.
best_class_model_v1 = mean_results.loc["Version I", "F1"].idxmax()
best_class_model_v2 = mean_results.loc["Version II", "F1"].idxmax()

print("Threshold-analysis model — Version I:", best_class_model_v1)
print("Threshold-analysis model — Version II:", best_class_model_v2)

oof_prob_v1 = class_oof_v1[best_class_model_v1]["prob"]
oof_fold_v1 = class_oof_v1[best_class_model_v1]["fold"]
oof_prob_v2 = class_oof_v2[best_class_model_v2]["prob"]
oof_fold_v2 = class_oof_v2[best_class_model_v2]["fold"]

def threshold_table(y, prob):
    rows = []
    for threshold in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
        pred = (prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
        rows.append({
            "Threshold": threshold,
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp
        })
    return pd.DataFrame(rows)

threshold_v1 = threshold_table(yclass_v1, oof_prob_v1)
threshold_v2 = threshold_table(yclass_v2, oof_prob_v2)

print(f"Version I {best_class_model_v1}: ROC-AUC={roc_auc_score(yclass_v1, oof_prob_v1):.3f}, PR-AUC={average_precision_score(yclass_v1, oof_prob_v1):.3f}")
display(threshold_v1.round(3))
print(f"Version II {best_class_model_v2}: ROC-AUC={roc_auc_score(yclass_v2, oof_prob_v2):.3f}, PR-AUC={average_precision_score(yclass_v2, oof_prob_v2):.3f}")
display(threshold_v2.round(3))


In [ ]:
for title, y, prob in [
    (f"Version I — {best_class_model_v1}", yclass_v1, oof_prob_v1),
    (f"Version II — {best_class_model_v2}", yclass_v2, oof_prob_v2)
]:
    thresholds = np.linspace(0.10, 0.80, 29)
    precision, recall, f1s = [], [], []
    for t in thresholds:
        pred = (prob >= t).astype(int)
        precision.append(precision_score(y, pred, zero_division=0))
        recall.append(recall_score(y, pred, zero_division=0))
        f1s.append(f1_score(y, pred, zero_division=0))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(thresholds, precision, label="Precision")
    ax.plot(thresholds, recall, label="Recall")
    ax.plot(thresholds, f1s, label="F1")
    label_line_ends(ax)
    ax.set_title(title + " — Threshold Trade-off")
    ax.set_xlabel("Classification threshold")
    ax.set_ylabel("Score")
    ax.legend()
    plt.tight_layout()
    plt.show()


### Threshold-dependent precision–recall interpretation

This analysis answers a practical question: **at what probability threshold should a condition be labelled toxic, given the cost of missed toxic cases versus false alarms?**

The default threshold of 0.50 is a decision rule, not a biological constant.

- Lowering the threshold usually predicts more cases as toxic: **recall tends to increase**, but precision can fall because more nontoxic cases are flagged.
- Raising the threshold usually produces fewer toxic predictions: **precision may increase**, but recall can fall because more toxic cases are missed.
- In an imbalanced toxicity-screening setting, this trade-off matters because the preferred threshold depends on whether the priority is to catch as many potentially toxic conditions as possible or to reduce unnecessary follow-up experiments.

The notebook therefore reports precision, recall and F1 across multiple thresholds using **OOF probabilities**. It does not claim that the F1-maximizing threshold is universally optimal; the final operating point should reflect the intended screening cost and would ideally be confirmed on independent data.


In [ ]:
# Select the lowest-MAE regression model in each dataset for fold-stability analysis.
best_reg_model_v1 = regression_means.loc["Version I", "MAE"].idxmin()
best_reg_model_v2 = regression_means.loc["Version II", "MAE"].idxmin()

reg_oof_v1 = reg_oof_all_v1[best_reg_model_v1]["pred"]
reg_fold_v1 = reg_oof_all_v1[best_reg_model_v1]["fold"]
reg_oof_v2 = reg_oof_all_v2[best_reg_model_v2]["pred"]
reg_fold_v2 = reg_oof_all_v2[best_reg_model_v2]["fold"]

def fold_regression_table(y, pred, fold_id):
    rows = []
    for fold in sorted(np.unique(fold_id)):
        m = fold_id == fold
        rows.append({
            "Fold": fold,
            "MAE": mean_absolute_error(y[m], pred[m]),
            "RMSE": mean_squared_error(y[m], pred[m]) ** 0.5,
            "R2": r2_score(y[m], pred[m])
        })
    return pd.DataFrame(rows)

fold_reg_v1 = fold_regression_table(yreg_v1.to_numpy(), reg_oof_v1, reg_fold_v1)
fold_reg_v2 = fold_regression_table(yreg_v2.to_numpy(), reg_oof_v2, reg_fold_v2)

print(f"Version I — {best_reg_model_v1} fold stability")
display(fold_reg_v1.round(3))
print(f"Version II — {best_reg_model_v2} fold stability")
display(fold_reg_v2.round(3))

stability_summary = pd.DataFrame({
    "Version I": [fold_reg_v1["MAE"].std(), fold_reg_v1["R2"].std(), fold_reg_v1["R2"].min(), fold_reg_v1["R2"].max()],
    "Version II": [fold_reg_v2["MAE"].std(), fold_reg_v2["R2"].std(), fold_reg_v2["R2"].min(), fold_reg_v2["R2"].max()],
}, index=["MAE std across folds", "R2 std across folds", "Minimum fold R2", "Maximum fold R2"])
print("Fold-to-fold variability summary")
display(stability_summary.round(3))


In [ ]:
for title, y, pred in [
    (f"Version I — {best_reg_model_v1}", yreg_v1.to_numpy(), reg_oof_v1),
    (f"Version II — {best_reg_model_v2}", yreg_v2.to_numpy(), reg_oof_v2)
]:
    residual = y - pred
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(y, pred, alpha=0.30, color=CHART_COLORS[0], edgecolors="none")
    annotate_sample_count(ax, len(y))
    lo = min(np.min(y), np.min(pred)); hi = max(np.max(y), np.max(pred))
    ax.plot([lo, hi], [lo, hi])
    ax.set_title(title + " — Observed vs Predicted Viability")
    ax.set_xlabel("Observed viability (%)")
    ax.set_ylabel("Predicted viability (%)")
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(pred, residual, alpha=0.30, color=CHART_COLORS[1], edgecolors="none")
    annotate_sample_count(ax, len(y))
    ax.axhline(0)
    ax.set_title(title + " — Regression Residuals")
    ax.set_xlabel("Predicted viability (%)")
    ax.set_ylabel("Residual (observed - predicted)")
    plt.tight_layout(); plt.show()


### Fold-level regression stability interpretation

Fold-level stability asks: **does the selected regression model perform similarly when different PubMed studies are left out?**

The notebook reports MAE, RMSE and R² separately for each fold, plus their mean, standard deviation, minimum and maximum.

- **Small variation across folds** suggests that performance is relatively stable across the sampled study groups.
- **Large variation across folds** suggests sensitivity to which studies are used for training and validation.

Substantial fold-to-fold variation can reflect:
- dataset heterogeneity across laboratories, assays, materials or cell systems,
- small effective sample size once observations are grouped by study,
- model instability,
- or distribution shift between source studies.

This analysis does not turn cross-validation into an independent test set; it diagnoses how dependent performance is on the particular train–validation study split.


# Task G — Comparative Analysis and Interpretation

Task G synthesizes the **entire notebook** rather than giving a high-level or generic comparison.

It explicitly compares:
1. all classification models and their principal metrics,
2. all regression models and their MAE, RMSE and R²,
3. model stability across grouped folds,
4. Version I versus Version II performance,
5. preprocessing/curation differences that may help explain changes,
6. which preprocessing decisions appear useful,
7. and which limitations remain unresolved.

The purpose is to answer: **what was done, why was it done, what was observed, and what decision follows from that observation?**

Because the two versions are related processing stages of the same literature resource, differences are interpreted as evidence about the effect of curation/preprocessing within this resource, not as external validation.


In [ ]:
# Classification synthesis
class_synthesis_rows = []
for dataset in ["Version I", "Version II"]:
    d = mean_results.loc[dataset]
    class_synthesis_rows.append({
        "Dataset": dataset,
        "Best F1 model": d["F1"].idxmax(),
        "Best F1": d["F1"].max(),
        "Best Recall model": d["Recall"].idxmax(),
        "Best Recall": d["Recall"].max(),
        "Best ROC-AUC model": d["ROC-AUC"].idxmax(),
        "Best ROC-AUC": d["ROC-AUC"].max(),
        "Best PR-AUC model": d["PR-AUC"].idxmax(),
        "Best PR-AUC": d["PR-AUC"].max(),
    })
classification_synthesis = pd.DataFrame(class_synthesis_rows).set_index("Dataset")
print("Classification synthesis")
display(classification_synthesis.round(3))

# Regression synthesis
reg_synthesis_rows = []
for dataset in ["Version I", "Version II"]:
    d = regression_means.loc[dataset]
    reg_synthesis_rows.append({
        "Dataset": dataset,
        "Lowest-MAE model": d["MAE"].idxmin(),
        "Lowest MAE": d["MAE"].min(),
        "Lowest-RMSE model": d["RMSE"].idxmin(),
        "Lowest RMSE": d["RMSE"].min(),
        "Highest-R2 model": d["R2"].idxmax(),
        "Highest R2": d["R2"].max(),
    })
regression_synthesis = pd.DataFrame(reg_synthesis_rows).set_index("Dataset")
print("Regression synthesis")
display(regression_synthesis.round(3))

# Like-for-like classification changes from V1 to V2.
class_delta_rows = []
for model in mean_results.index.get_level_values("Model").unique():
    v1 = mean_results.loc[("Version I", model)]
    v2 = mean_results.loc[("Version II", model)]
    class_delta_rows.append({
        "Model": model,
        "F1 change V2-V1": v2["F1"] - v1["F1"],
        "Recall change V2-V1": v2["Recall"] - v1["Recall"],
        "ROC-AUC change V2-V1": v2["ROC-AUC"] - v1["ROC-AUC"],
        "PR-AUC change V2-V1": v2["PR-AUC"] - v1["PR-AUC"],
    })
classification_version_change = pd.DataFrame(class_delta_rows).set_index("Model").round(3)
print("Classification change: Version II minus Version I")
display(classification_version_change)

print("Regression change: Version II minus Version I")
display(regression_version_change)


## Integrated interpretation

The synthesis tables above identify the strongest models rather than relying on visual impressions alone.

- **Classification:** the best model is reported separately for F1, recall, ROC-AUC and PR-AUC because “best” depends on the screening objective. F1 around 0.45–0.50 remains modest and is interpreted in light of class imbalance, heterogeneity, missingness and limited feature informativeness.
- **Regression:** the lowest-MAE, lowest-RMSE and highest-R² models are identified explicitly. MAE is interpreted in viability percentage points, and low or negative R² values are treated as evidence of limited predictive signal rather than ignored.
- **Version I vs Version II:** numerical deltas show whether Version II lowers MAE/RMSE or raises R² and whether classification metrics change. These differences are then considered against the data-quality and preprocessing observations from Tasks A–D.
- **Stability:** fold-level variability is used to judge whether reported averages are consistent across unseen source-study groups.
- **Preprocessing:** median numerical imputation, explicit categorical missing levels, one-hot encoding, evidence-based log transformation, model-appropriate scaling and group-aware validation each have a stated reason. These steps improve methodological validity but do not guarantee high predictive performance.
- **Unresolved limitations:** literature heterogeneity, unmeasured biological/experimental factors, class imbalance, sparse categories and lack of an independent external test set remain important constraints.

### Final scientific conclusion

The notebook should therefore be read as a **carefully validated baseline modelling study**, not as a deployable clinical or laboratory decision system. The most defensible result is the combination of model performance, its uncertainty/stability, and the limitations revealed by the analysis.


## Limitations

1. Literature-derived observations are heterogeneous across laboratories, assays, cell systems and exposure protocols.
2. Toxic and nontoxic classes are imbalanced, which makes minority-class prediction difficult.
3. Some materials, assays and cell systems are much more represented than others.
4. Viability contains extreme values and assay-dependent variation; not every extreme observation is necessarily an error.
5. The available descriptors may omit relevant physicochemical or biological factors, limiting achievable F1 and R².
6. High-cardinality categorical variables create sparse representations and some rare categories have limited evidence.
7. Version II overlaps with Version I and is therefore not an independent external validation set.
8. Grouped cross-validation reduces study leakage but cannot eliminate every form of similarity among related materials or experimental conditions.
9. Fold-to-fold variation indicates source-study heterogeneity and sensitivity to the train/validation split.
10. The model comparison is focused rather than an exhaustive hyperparameter search.

These limitations explain why modest F1/R² values should be interpreted scientifically rather than presented without context. Future work should include genuinely external experimental data, broader feature coverage and prospective laboratory validation.


# One-Page Submission Summary

This summary condenses the main finding from each stage of the analysis.

- **Task A — EDA:** the feature space is heterogeneous in scale, distribution, categorical cardinality and class balance. These observations motivate feature-specific preprocessing rather than a single normalization rule.
- **Task B — Missingness/data quality:** missingness is variable across predictors and provenance fields. Numerical scientific predictors are median-imputed inside training folds; categorical missingness is represented explicitly; provenance/quality metadata are not used as ordinary exposure predictors.
- **Task C — Prediction problems:** toxicity classification and viability regression are treated as separate targets; the alternate endpoint is excluded to prevent target leakage.
- **Task D — Preprocessing:** right-skewed non-negative variables may receive `log1p`; scale-sensitive models receive standardization; tree models do not require numerical scaling; categorical features are one-hot encoded with rare/unseen-category handling.
- **Task E — Classification:** Logistic Regression, KNN, Random Forest and XGBoost provide linear, distance-based and nonlinear comparisons. Modest F1 results are interpreted rather than hidden by accuracy.
- **Task E — Regression:** Ridge, Random Forest and XGBoost are compared. MAE/RMSE are interpreted in viability percentage points and R² is interpreted relative to the mean-prediction baseline.
- **Task F — Validation:** group-aware 5-fold OOF cross-validation keeps each PubMed study out of the training data for the fold in which it is predicted. No independent held-out test set is claimed.
- **Task G — Comparison:** model winners, Version I/II changes, fold stability, preprocessing implications and unresolved limitations are synthesized explicitly.

**Overall:** the strongest-performing approaches should be read together with their stability and limitations. The analysis supports reproducible, leakage-resistant baseline conclusions while showing that the available literature-derived features do not fully explain cytotoxicity or cell-viability variation.


# Reproducibility Statement

- Random seed: **42**
- Both supplied curated metal-oxide datasets are analysed independently using the same feature definitions and leakage safeguards.
- `Pubmed ID` is retained only for group-aware validation and is excluded from predictors.
- Classification excludes `Viability (%)`; regression excludes `Toxicity`.
- Preprocessing is embedded in scikit-learn pipelines so imputation, transformation, scaling and encoding are fit on training folds only.
- Numerical transformation decisions are documented from EDA; categorical handling is explicit.
- Library versions and Python/platform information are printed at the beginning of the notebook.
- Every reported table and figure is generated by executable code cells.

## Final conclusion

This assignment demonstrates a complete, reasoning-led and leakage-aware machine-learning workflow for metal-oxide nanotoxicology. The analysis connects EDA to preprocessing, preprocessing to model choice, validation to interpretation, and Version I/Version II differences to the question of data curation. Model outputs should be treated as screening/prioritization evidence rather than substitutes for experimental validation.

**Execution design:** Task F reuses out-of-fold predictions generated during Task E rather than refitting the same candidate models. This reduces runtime while preserving the same study-aware validation predictions.
